# FMI CloudCast → GK2A 겨울 파인튜닝 (Step 1 게이트) — v2

**목적**: 겨울(1월) 운량 나우캐스트에서 광류 이류(M1)가 실패하는 구간을 DL로 보완할 수 있는지 판정.

**준비물** (Google Drive `MyDrive/nwp_dl/`에 업로드):
- `fmi_cloudcast_unet.tar.gz` (사전학습 가중치, 340MB)
- `dataset/*.npz` (dl_dataset.py 산출, 파일명 YYYYMMDD.npz — 12월+1월 62개)

**모델 규격** (가중치 디렉토리명·FMI 코드 실측): `hist=4, lc=12(oh=False→k/12 스칼라 평면),
sun=True(고도각 도 단위, 프레임별 min-max), dt/topo/terrain=False` → **입력 6채널 = [hist4 | k/12 | sun]**

**규약**: 12월=학습, 1월=검증(학습 미사용). 런타임 → GPU(T4) 선택 후 전체 실행.
라이선스 미명시 자료이므로 개인 연구 용도로만 사용.

In [ ]:
# Keras 3는 구형 SavedModel 미지원 → tf_keras(Keras 2 호환)로 로드/학습
!pip install -q tf_keras
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/nwp_dl'
import os, glob, tarfile, math, datetime as dt
import numpy as np
import tensorflow as tf
import tf_keras
print('TF', tf.__version__, '| tf_keras', tf_keras.__version__, '| GPU:', tf.config.list_physical_devices("GPU"))

In [ ]:
# 1) 가중치 압축 해제 + tf_keras 로드
MODEL_DIR = '/content/fmi_model'
if not glob.glob(f'{MODEL_DIR}/**/saved_model.pb', recursive=True):
    os.makedirs(MODEL_DIR, exist_ok=True)
    with tarfile.open(f'{BASE}/fmi_cloudcast_unet.tar.gz') as t:
        t.extractall(MODEL_DIR)
cands = [r for r, d, f in os.walk(MODEL_DIR) if 'saved_model.pb' in f]
print('SavedModel:', cands)
model = tf_keras.models.load_model(cands[0], compile=False)
N_CH = int(model.inputs[0].shape[-1])
print('입력:', model.inputs[0].shape)
assert N_CH == 6, f'예상(6채널=[hist4|k/12|sun])과 다름: {N_CH} — 중단, 채널 구성 재확인 필요'

In [ ]:
# 2) 데이터 로드 (uint8 0..100, 255=결측)
def load_month(pattern):
    stamps, frames = [], []
    for f in sorted(glob.glob(f'{BASE}/dataset/{pattern}.npz')):
        z = np.load(f)
        stamps += list(z['stamps'])
        frames.append(z['frames'])
    return np.array(stamps), np.concatenate(frames)

st_tr, fr_tr = load_month('202512??')
st_va, fr_va = load_month('202601??')
print('학습(12월):', fr_tr.shape, ' 검증(1월):', fr_va.shape)
IDX_TR = {s: i for i, s in enumerate(st_tr)}
IDX_VA = {s: i for i, s in enumerate(st_va)}

In [ ]:
# 3) 태양고도 채널 — FMI 방식 실측: 고도각(도) 계산 후 프레임별 min-max 0..1
#    (preprocess.py create_sun_elevation_angle → preprocess_single normalize=true)
LATS = np.linspace(46.0, 29.7, 512)[:, None] * np.ones((1, 512))
LONS = np.ones((512, 1)) * np.linspace(113.0, 139.5, 512)[None, :]

def sun_channel(stamp):
    t = dt.datetime.strptime(stamp, '%Y%m%d%H%M')
    doy = t.timetuple().tm_yday
    decl = -23.44 * math.cos(math.radians(360/365*(doy+10)))
    hour = t.hour + t.minute/60
    ha = (hour*15 - 180) + LONS  # 시간각(deg), UTC
    sin_el = (np.sin(np.radians(LATS))*math.sin(math.radians(decl)) +
              np.cos(np.radians(LATS))*math.cos(math.radians(decl))*np.cos(np.radians(ha)))
    el = np.degrees(np.arcsin(np.clip(sin_el, -1, 1)))
    if el.max() > el.min():
        el = (el - el.min()) / np.ptp(el)
    return el.astype(np.float32)

In [ ]:
# 4) 샘플 생성기 — X=[hist4 | k/12 평면 | sun(target)] , y=target(+10*(k+1)분)
STEP, N_HIST, N_LC = 10, 4, 12

def make_sample(frames, idx, s0, k):
    t0 = dt.datetime.strptime(s0, '%Y%m%d%H%M')
    need = [(t0 - dt.timedelta(minutes=STEP*i)).strftime('%Y%m%d%H%M') for i in range(N_HIST-1, -1, -1)]
    need.append((t0 + dt.timedelta(minutes=STEP*(k+1))).strftime('%Y%m%d%H%M'))
    if not all(n in idx for n in need):
        return None
    arrs = [frames[idx[n]] for n in need]
    if any((a == 255).mean() > 0.1 for a in arrs):
        return None
    hist = [np.where(a == 255, 50, a).astype(np.float32)/100.0 for a in arrs[:N_HIST]]
    y = np.where(arrs[-1] == 255, 50, arrs[-1]).astype(np.float32)/100.0
    lt = np.full((512, 512), k / N_LC, np.float32)
    X = np.stack(hist + [lt, sun_channel(need[-1])], -1)
    return X, y[..., None]

def gen(frames, idx, shuffle=True):
    keys = list(idx.keys())
    while True:
        order = np.random.permutation(len(keys)) if shuffle else range(len(keys))
        for i in order:
            s = make_sample(frames, idx, keys[i], int(np.random.randint(N_LC)))
            if s is not None:
                yield s

sig = (tf.TensorSpec((512, 512, 6), tf.float32), tf.TensorSpec((512, 512, 1), tf.float32))
ds_tr = tf.data.Dataset.from_generator(lambda: gen(fr_tr, IDX_TR), output_signature=sig).batch(4).prefetch(2)
ds_va = tf.data.Dataset.from_generator(lambda: gen(fr_va, IDX_VA, shuffle=False), output_signature=sig).batch(4).take(200)

In [ ]:
# 5) 손실(bc+l1, FMI bcl1 방식) + 파인튜닝  (T4 대략 2~4시간)
def bcl1(y_true, y_pred):
    bc = tf_keras.losses.binary_crossentropy(y_true, y_pred)
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred), axis=-1)
    return bc + l1

model.compile(optimizer=tf_keras.optimizers.Adam(1e-5), loss=bcl1, metrics=['mae'])
hist = model.fit(ds_tr, steps_per_epoch=1500, epochs=3, validation_data=ds_va)
# 세션이 끊기면: 런타임 재시작 후 위 셀들 재실행 → epochs를 1~2로 줄여 재개

In [ ]:
# 6) 저장 — 단일 h5 (로컬 tf_keras 로드용)
out = f'{BASE}/gk2a_finetuned.h5'
model.save(out)
print('완료:', out, round(os.path.getsize(out)/2**20), 'MB')
print('→ PC의 kpx-model-charts/dl/ 에 gk2a_finetuned.h5 내려받으세요')

In [ ]:
# 7) (선택) 1월 사례 눈검증 — +60분(k=5) 예측 vs 실제
import matplotlib.pyplot as plt
smp = make_sample(fr_va, IDX_VA, st_va[len(st_va)//2], 5)
if smp:
    X, y = smp
    p = model.predict(X[None])[0, ..., 0]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, img, t in zip(ax, [X[..., 3], p, y[..., 0]], ['input t', 'pred +60min', 'actual']):
        a.imshow(img, cmap='gray', vmin=0, vmax=1); a.set_title(t); a.axis('off')
    plt.show()